In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

Fine-Tuning Qwen3-4B with Unsloth & QLoRA


In [7]:
%%capture
import os

!pip install pip3-autoremove
!pip install torch torchvision torchaudio xformers --index-url https://download.pytorch.org/whl/cu128
!pip install unsloth
!pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2
!pip install groq

#### Generate the Dataset with Groq¶


In [3]:
import asyncio, json, random
from groq import AsyncGroq
from kaggle_secrets import UserSecretsClient

client = AsyncGroq(api_key=UserSecretsClient().get_secret("GROQ_API_KEY"))

TOPICS = [
    "Python basics and data structures", "pandas and data cleaning",
    "machine learning fundamentals", "overfitting and regularization",
    "neural networks and backpropagation", "transformers and attention",
    "LLMs and how they are trained", "prompt engineering",
    "RAG and vector databases", "fine-tuning and LoRA",
    "AI agents and tool calling", "model evaluation and metrics",
    "APIs and deployment basics", "SQL and databases",
    "statistics for data science", "career advice for AI engineers",
]

SYSTEM_PROMPT = """You generate training data for 'ProTutor', an expert AI engineering tutor.
Each example is a question a beginner software developer would ask, plus ProTutor's answer.
Every answer MUST follow these 3 rules:
1. Written in clear, professional, and easy-to-understand standard English.
2. Explains the concept using a simple, universal real-world analogy (e.g., a library, a kitchen, a post office).
3. Ends with a clear one-line summary starting with 'Key Takeaway:'.
Keep answers 60-120 words. Vary question phrasing and difficulty.
Return JSON: {"examples": [{"user": "...", "assistant": "..."}, ...]}"""

async def generate_batch(topic: str, n: int = 20, retries: int = 3) -> list[dict]:
    """Ask the teacher model for n Q&A pairs about one topic. Backs off on rate limits. this will take topics one by one"""
    for attempt in range(retries):
        try:
            resp = await client.chat.completions.create(
                model="openai/gpt-oss-120b",
                messages=[{"role": "system", "content": SYSTEM_PROMPT},
                          {"role": "user", "content": f"Generate {n} question-answer pairs about: {topic}"}],
                response_format={"type": "json_object"},
                temperature=0.9,
            )
            return json.loads(resp.choices[0].message.content)["examples"]
        except Exception as e:
            print(f" retry {attempt + 1} for '{topic}': {e}")
            await asyncio.sleep(20 * (attempt + 1))
    return []






In [5]:

async def build_dataset() -> list[dict]:
    """Generate all topics, 2 at a time - free tier allows 8,000 tokens/min, one call uses ~3,300."""
    pairs = []
    for i in range(0, len(TOPICS), 2):
        results = await asyncio.gather(*[generate_batch(t) for t in TOPICS[i:i + 2]])
        for batch in results:
            pairs.extend(batch)
        print(f"{len(pairs)} examples generated")
    return pairs

pairs = await build_dataset()
random.shuffle(pairs)

train_pairs, heldout_pairs = pairs[:-20], pairs[-20:]

with open("/kaggle/working/desitutor_data.json", "w") as f:
    json.dump({"train": train_pairs, "heldout": heldout_pairs}, f, ensure_ascii=False, indent=2)

43 examples generated
86 examples generated
 retry 1 for 'neural networks and backpropagation': Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gpt-oss-120b` in organization `org_01m0tgvedmenk8pvkgtpwpck30` service tier `on_demand` on tokens per minute (TPM): Limit 8000, Used 7967, Requested 3266. Please try again in 24.2475s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}
131 examples generated
176 examples generated
217 examples generated
 retry 1 for 'model evaluation and metrics': Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gpt-oss-120b` in organization `org_01m0tgvedmenk8pvkgtpwpck30` service tier `on_demand` on tokens per minute (TPM): Limit 8000, Used 4867, Requested 3264. Please try again in 982.5ms. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_l

In [4]:

file_path = "/kaggle/input/datasets/preatcher/desitutor-data/desitutor_data.json"

# 2. Open and load the JSON file
with open(file_path, "r", encoding="utf-8") as f:
    data = json.load(f)

# 3. Extract the lists from the dictionary
train_pairs = data["train"]
heldout_pairs = data["heldout"]


In [5]:
print(f"Train: {len(train_pairs)}, Held-out: {len(heldout_pairs)}")
print(json.dumps(train_pairs[0], ensure_ascii=False, indent=2))

Train: 322, Held-out: 20
{
  "user": "What is a stored procedure and when should I use it?",
  "assistant": "A stored procedure is like a pre‑written recipe stored in the kitchen pantry: you can call it whenever you need the same set of steps, without rewriting them each time. It’s a batch of SQL statements saved on the server, which can accept parameters, contain logic, and return results. Use it for repetitive tasks, complex business logic, or to centralize security. Key Takeaway: A stored procedure is a reusable, server‑side SQL script that encapsulates logic for consistent, efficient execution."
}


### Load Qwen3-4B in 4-bit and Attach LoRA¶


In [8]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen3-4B-Instruct-2507",
    max_seq_length = 2048,
    load_in_4bit = True,
    load_in_8bit = False,
    full_finetuning = False,
)


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.8.19: Fast Qwen3 patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


In [9]:
from unsloth import FastLanguageModel

FastLanguageModel.for_inference(model)

Qwen3ForCausalLM(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 2560, padding_idx=151654)
    (layers): ModuleList(
      (0): Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear(in_features=2560, out_features=4096, bias=False)
          (k_proj): Linear(in_features=2560, out_features=1024, bias=False)
          (v_proj): Linear(in_features=2560, out_features=1024, bias=False)
          (o_proj): Linear(in_features=4096, out_features=2560, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (rotary_emb): LlamaRotaryEmbedding()
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear4bit(in_features=2560, out_features=9728, bias=False)
          (up_proj): Linear4bit(in_features=2560, out_features=9728, bias=False)
          (down_proj): Linear4bit(in_features=9728, out_features=2560, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): Q

In [11]:
test_prompts = [
    "What is the capital of France?",

    "What is 27 * 43?",

    "Explain what a Python decorator is.",

    "Why does this Python code have O(n²) complexity?",

    "Find the bug in this code and explain how to fix it.",

    "A train travels 60 km in 45 minutes. What is its average speed?",

    "You have 12 balls and one is heavier. Find the heavier ball using a balance scale in 3 weighings.",

    "Design a scalable RAG architecture capable of handling 10,000 concurrent users.",

    "Why might a distributed system return stale data intermittently?",

    "Explain the difference between LoRA and full fine-tuning."
]
baseline_responses = []

for prompt in test_prompts:

    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
    ).to(model.device)

    outputs = model.generate(
        input_ids=inputs,
        max_new_tokens=512,
        do_sample=False,
    )
    response = tokenizer.decode(
        outputs[0][inputs.shape[-1]:],
        skip_special_tokens=True,
    )

    baseline_responses.append({
        "prompt": prompt,
        "response": response,
    })

    print("=" * 80)
    print("PROMPT:")
    print(prompt)
    print("\nRESPONSE:")
    print(response)

PROMPT:
What is the capital of France?

RESPONSE:
The capital of France is Paris.
PROMPT:
What is 27 * 43?

RESPONSE:
Let's calculate $ 27 \times 43 $ step by step.

We can use the distributive property:

$$
27 \times 43 = 27 \times (40 + 3) = (27 \times 40) + (27 \times 3)
$$

Now compute each part:

- $ 27 \times 40 = 27 \times 4 \times 10 = 108 \times 10 = 1080 $
- $ 27 \times 3 = 81 $

Now add them:

$$
1080 + 81 = 1161
$$

✅ So, $ 27 \times 43 = \boxed{1161} $
PROMPT:
Explain what a Python decorator is.

RESPONSE:
A **Python decorator** is a special function that allows you to modify or enhance the behavior of another function without directly changing its code. It works by wrapping a function and adding extra functionality—like logging, timing, authentication, or caching—before or after the original function executes.

### How Decorators Work

Decorators are implemented using **higher-order functions**—functions that take other functions as arguments and can return functions.

He

In [12]:

model = FastLanguageModel.get_peft_model(
    model,
    r = 32,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", 
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 32,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

Unsloth 2026.8.19 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


### Format the Data with the Chat Template¶


In [36]:
from unsloth.chat_templates import get_chat_template
from datasets import Dataset

tokenizer = get_chat_template(tokenizer, chat_template = "qwen3-instruct")

def to_conversation(pair: dict) -> dict:
    """Wrap one Q&A pair in the role/content structure chat templates expect."""
    return {"conversations": [
        {"role": "user", "content": pair["user"]},
        {"role": "assistant", "content": pair["assistant"]},
    ]}

def formatting_prompts_func(examples: dict) -> dict:
    """Serialize each conversation into one ChatML training string."""
    texts = [tokenizer.apply_chat_template(c, tokenize = False, add_generation_prompt = False) 
             for c in examples["conversations"]]
    return {"text": texts}



In [37]:
dataset = Dataset.from_list([to_conversation(p) for p in train_pairs])


In [38]:
print(dataset[0])

{'conversations': [{'role': 'user', 'content': 'What is a stored procedure and when should I use it?'}, {'role': 'assistant', 'content': 'A stored procedure is like a pre‑written recipe stored in the kitchen pantry: you can call it whenever you need the same set of steps, without rewriting them each time. It’s a batch of SQL statements saved on the server, which can accept parameters, contain logic, and return results. Use it for repetitive tasks, complex business logic, or to centralize security. Key Takeaway: A stored procedure is a reusable, server‑side SQL script that encapsulates logic for consistent, efficient execution.'}]}


In [39]:
dataset = dataset.map(formatting_prompts_func, batched = True)
print(dataset[0]["text"])

Map:   0%|          | 0/322 [00:00<?, ? examples/s]

<|im_start|>user
What is a stored procedure and when should I use it?<|im_end|>
<|im_start|>assistant
A stored procedure is like a pre‑written recipe stored in the kitchen pantry: you can call it whenever you need the same set of steps, without rewriting them each time. It’s a batch of SQL statements saved on the server, which can accept parameters, contain logic, and return results. Use it for repetitive tasks, complex business logic, or to centralize security. Key Takeaway: A stored procedure is a reusable, server‑side SQL script that encapsulates logic for consistent, efficient execution.<|im_end|>



In [40]:
from transformers import TextStreamer

def ask(question: str, max_new_tokens: int = 200) -> str:
    """Generate one answer from the model in its current state."""
    messages = [{"role": "user", "content": question}]
    text = tokenizer.apply_chat_template(messages, tokenize = False, add_generation_prompt = True)
    inputs = tokenizer(text, return_tensors = "pt").to("cuda")
    out = model.generate(**inputs, max_new_tokens = max_new_tokens, 
                         temperature = 0.7, top_p = 0.8, top_k = 20)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens = True)

heldout_questions = [p["user"] for p in heldout_pairs]
base_answers = [ask(q) for q in heldout_questions]

print("Q:", heldout_questions[0])
print("BASE MODEL:", base_answers[0][:400])

Q: What does the SELECT statement do in SQL?
BASE MODEL: The **SELECT statement** in SQL is used to **retrieve data** from one or more tables in a database.

### Key Functions of the SELECT Statement:
- **Queries data**: It allows you to specify which columns (or expressions) you want to retrieve.
- **Filters data**: Using clauses like `WHERE`, you can filter rows based on certain conditions.
- **Sorts data**: Using `ORDER BY`, you can sort the results 


In [41]:
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    eval_dataset = None,
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60,
        learning_rate = 2e-4,
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 3407,
        report_to = "none",
    ),
)

trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|im_start|>user\n",
    response_part = "<|im_start|>assistant\n",
)

stats = trainer.train()

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/322 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


Map:   0%|          | 0/322 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 322 | Num Epochs = 2 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 66,060,288 of 4,088,528,384 (1.62% trained)


Step,Training Loss
1,2.880400
2,2.853600
3,2.904600
4,2.479800
5,2.187100
6,2.140800
7,2.019400
8,1.878000
9,1.750400
10,1.641100


In [43]:
finetuned_responses = []

for prompt in test_prompts:

    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
    ).to(model.device)

    outputs = model.generate(
        input_ids=inputs,
        max_new_tokens=512,
        do_sample=False,
    )

    response = tokenizer.decode(
        outputs[0][inputs.shape[-1]:],
        skip_special_tokens=True,
    )

    finetuned_responses.append({
        "prompt": prompt,
        "response": response,
    })

    print("=" * 80)
    print("PROMPT:")
    print(prompt)

    print("\nFINE-TUNED RESPONSE:")
    print(response)

PROMPT:
What is the capital of France?

FINE-TUNED RESPONSE:
Paris is the capital of France, much like the city of New York is the capital of the United States. It is the country’s political, cultural, and economic hub, home to the French government, the Louvre Museum, and many international organizations. Key Takeaway: Paris is the capital of France, serving as its main center for government, culture, and commerce.
PROMPT:
What is 27 * 43?

FINE-TUNED RESPONSE:
Think of 27 as 20 + 7 and 43 as 40 + 3. Multiply each pair: (20×40)=800, (20×3)=60, (7×40)=280, (7×3)=21. Add them: 800 + 60 + 280 + 21 = 1161. So, 27 × 43 = 1161. Key Takeaway: Break numbers into parts, multiply each pair, then sum the results.
PROMPT:
Explain what a Python decorator is.

FINE-TUNED RESPONSE:
A decorator is like a frame around a picture: it wraps the original picture (function) and adds extra features (side effects) without changing the picture itself. In Python, a decorator is a function that takes another fu